# G-Eval & Custom Criteria

This notebook covers DeepEval's most flexible evaluation tools:

1. **G-Eval** — Chain-of-thought evaluation with custom criteria, evaluation steps, and rubrics
2. **Custom Metrics** — Build your own deterministic and LLM-based metrics

### Prerequisites
- Notebook 02 (pipeline results in `data/pipeline_results.json`)
- Notebook 03-04 (familiarity with DeepEval/RAGAS retriever and generator metrics)

-
## 1. Setup & Imports

In [ ]:
import os
import json
from dotenv import load_dotenv

dotenv_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
if not os.path.exists(dotenv_path):
    dotenv_path = os.path.join(os.getcwd(), ".env")
load_dotenv(dotenv_path, override=True)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
print("Environment loaded.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from deepeval.test_case import LLMTestCase
from deepeval import evaluate

print("Base imports successful.")

-
## 2. Load Test Data

We load the pipeline results from Notebook 02, or use inline fallback data.

In [ ]:
import os
import json
data_path = os.path.join(os.getcwd(), "data", "pipeline_results.json")

if os.path.exists(data_path):
    with open(data_path) as f:
        pipeline_results = json.load(f)
    print(f"Loaded {len(pipeline_results)} results from file.")
else:
    pipeline_results = [
        {
            "query": "What is the return policy for regular items?",
            "expected_output": "Acme Corp offers a 30-day return policy. Items must be unused, in original packaging, with receipt.",
            "actual_output": "Acme Corp has a 30-day return policy. Items need to be unused and in original packaging with proof of purchase. Refunds are processed within 5-7 business days.",
            "retrieval_context": ["Acme Corp offers a 30-day return policy on all products. Items must be unused, in original packaging, with receipt. Refunds processed in 5-7 business days. Shipping costs are customer's responsibility unless item was defective."]
        },
        {
            "query": "How long do I have to return electronics?",
            "expected_output": "Electronics have a 15-day return window. Must include all original accessories. 15% restocking fee may apply.",
            "actual_output": "Electronic products have a 15-day return window, shorter than the standard 30 days. All original accessories, cables, manuals, and packaging must be included. A 15% restocking fee may apply to opened electronics.",
            "retrieval_context": ["Electronic products have a 15-day return window. All electronics must be returned with original accessories. A restocking fee of 15% may apply to opened electronics. Defective electronics can be exchanged within 90 days."]
        },
        {
            "query": "What shipping options are available?",
            "expected_output": "Standard (5-7 days, free over $50), Expedited (2-3 days, $12.99), Overnight (next day, $24.99).",
            "actual_output": "Acme Corp offers three shipping options: Standard Shipping (5-7 days, free for orders over $50), Expedited Shipping (2-3 days, $12.99), and Overnight Shipping (next business day, $24.99 if ordered before 2 PM EST).",
            "retrieval_context": ["Acme Corp offers Standard Shipping (5-7 business days, free over $50), Expedited Shipping (2-3 business days, $12.99), and Overnight Shipping ($24.99, next business day if ordered before 2 PM EST)."]
        },
        {
            "query": "What does the warranty cover?",
            "expected_output": "1-year limited warranty covering defects in materials and workmanship. Does not cover accidents.",
            "actual_output": "All Acme Corp products come with a 1-year limited warranty covering defects in materials and workmanship under normal use. It does not cover damage from accidents, misuse, or unauthorized modifications. Extended 2 and 3-year plans are available.",
            "retrieval_context": ["All Acme Corp branded products come with a 1-year limited warranty covering defects in materials and workmanship. Does not cover accidents, misuse, or unauthorized modifications. Extended warranty plans available."]
        },
        {
            "query": "What are the features of the SmartHome Hub?",
            "expected_output": "$149.99, WiFi/Bluetooth/Zigbee/Z-Wave, voice control, 5-inch touchscreen, energy monitoring, 10,000+ device compatibility.",
            "actual_output": "The Acme SmartHome Hub ($149.99) supports WiFi, Bluetooth, Zigbee, and Z-Wave. Features include voice control, 5-inch touchscreen, energy monitoring dashboard, and automated routines. Compatible with over 10,000 smart home devices.",
            "retrieval_context": ["The Acme SmartHome Hub ($149.99) supports WiFi, Bluetooth, Zigbee, and Z-Wave. Features: voice control, 5-inch touchscreen, energy monitoring, automated routines. Compatible with 10,000+ devices. 2-year warranty."]
        },
        {
            "query": "How can I contact customer support?",
            "expected_output": "Phone: 1-800-ACME-HELP (M-F 8-8 EST). Email: support@acmecorp.com. Live chat: M-Sat 9-6 EST.",
            "actual_output": "Contact support by phone at 1-800-ACME-HELP (Mon-Fri 8AM-8PM EST), email at support@acmecorp.com (24-48hr response), or live chat (Mon-Sat 9AM-6PM EST). Automated help center at help.acmecorp.com.",
            "retrieval_context": ["Customer support: Phone 1-800-ACME-HELP (M-F 8AM-8PM EST), Email support@acmecorp.com (24-48hr), Live chat M-Sat 9AM-6PM EST. Automated help center at help.acmecorp.com."]
        },
        {
            "query": "How does the loyalty program work?",
            "expected_output": "Free, 1 point per dollar. 100 points = $5 off. Silver (500+) and Gold (1000+) tiers with extra benefits.",
            "actual_output": "Acme Rewards is free to join. Earn 1 point per dollar spent, redeem 100 points for $5 off. Silver tier (500+ points/year) gets free expedited shipping. Gold tier (1000+ points/year) adds priority support. Points expire after 12 months of inactivity.",
            "retrieval_context": ["Acme Rewards: free to join, 1 point per dollar. 100 points = $5 off. Silver tier (500+): free expedited shipping. Gold tier (1000+): priority support. Points expire after 12 months inactivity."]
        },
        {
            "query": "Can I cancel my order?",
            "expected_output": "Within 1 hour via support or dashboard. After that, may not be possible. Shipped orders must be returned.",
            "actual_output": "Orders can be cancelled within 1 hour through customer support or your account dashboard. After 1 hour, the order enters processing and may not be cancellable. Shipped orders must go through the return process. Custom items cannot be cancelled once production begins.",
            "retrieval_context": ["Orders can be cancelled within 1 hour via customer support or account dashboard. After 1 hour, orders enter processing and may not be cancellable. Shipped orders must be returned. Custom items cannot be cancelled once production begins."]
        }
    ]
    print(f"Using {len(pipeline_results)} inline test cases.")

In [ ]:
# Create LLMTestCase objects
test_cases = []
for r in pipeline_results:
    tc = LLMTestCase(
        input=r["query"],
        actual_output=r["actual_output"],
        retrieval_context=r["retrieval_context"],
        expected_output=r["expected_output"],
    )
    test_cases.append(tc)

print(f"Created {len(test_cases)} test cases.")

-
## 3. G-Eval Metric

### How G-Eval Works

G-Eval is a framework-agnostic evaluation approach that uses an LLM as a judge with **chain-of-thought (CoT) reasoning**. The process:

1. You define a **custom evaluation criterion** (e.g., coherence, completeness).
2. The LLM generates a chain-of-thought reasoning about how well the output meets the criterion.
3. The LLM assigns a score (typically 1-5 or 1-10).
4. DeepEval normalizes the score to 0-1.

G-Eval is powerful because you can create any evaluation criterion you need, beyond the built-in metrics.

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

print("G-Eval imports successful.")

### 3.1 Coherence Criterion

Coherence measures whether the answer is well-structured, logically connected, and reads naturally.

In [ ]:
coherence_metric = GEval(
    name="Coherence",
    criteria="Evaluate the coherence of the actual output. A coherent response is well-structured, logically connected, uses proper transitions, and reads naturally without contradictions or abrupt topic changes.",
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],
    model="gpt-4o-mini",
    threshold=0.7,
)

print(f"G-Eval metric created: {coherence_metric.name}")

In [ ]:
# Run coherence on all test cases
coherence_scores = []

for i, tc in enumerate(test_cases):
    print(f"Evaluating {i+1}/{len(test_cases)}: {tc.input[:50]}...")
    coherence_metric.measure(tc)
    coherence_scores.append(coherence_metric.score)
    print(f"  Coherence: {coherence_metric.score:.4f}")

print(f"\nAverage Coherence: {np.mean(coherence_scores):.4f}")

### 3.2 Completeness Criterion

Completeness measures whether the answer covers all aspects of the question.

In [ ]:
completeness_metric = GEval(
    name="Completeness",
    criteria="Evaluate how completely the actual output answers the input question. A complete response addresses all aspects of the question, provides sufficient detail, and does not leave important parts unanswered. Compare with the expected output to assess coverage.",
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT,
    ],
    model="gpt-4o-mini",
    threshold=0.7,
)

completeness_scores = []

for i, tc in enumerate(test_cases):
    print(f"Evaluating {i+1}/{len(test_cases)}: {tc.input[:50]}...")
    completeness_metric.measure(tc)
    completeness_scores.append(completeness_metric.score)
    print(f"  Completeness: {completeness_metric.score:.4f}")

print(f"\nAverage Completeness: {np.mean(completeness_scores):.4f}")

### 3.3 Conciseness Criterion

Conciseness penalizes verbose, repetitive, or unnecessarily long answers.

In [ ]:
conciseness_metric = GEval(
    name="Conciseness",
    criteria="Evaluate how concise the actual output is. A concise response delivers the essential information without unnecessary filler, repetition, or excessive detail. It should be as brief as possible while still being complete and accurate.",
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],
    model="gpt-4o-mini",
    threshold=0.7,
)

conciseness_scores = []

for i, tc in enumerate(test_cases):
    print(f"Evaluating {i+1}/{len(test_cases)}: {tc.input[:50]}...")
    conciseness_metric.measure(tc)
    conciseness_scores.append(conciseness_metric.score)
    print(f"  Conciseness: {conciseness_metric.score:.4f}")

print(f"\nAverage Conciseness: {np.mean(conciseness_scores):.4f}")

### 3.4 Helpfulness Criterion

Helpfulness evaluates the overall practical utility of the response for the user.

In [ ]:
helpfulness_metric = GEval(
    name="Helpfulness",
    criteria="Evaluate how helpful the actual output would be to a customer asking the input question. A helpful response is actionable, easy to understand, provides clear next steps where appropriate, and makes the customer feel their question was fully addressed.",
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],
    model="gpt-4o-mini",
    threshold=0.7,
)

helpfulness_scores = []

for i, tc in enumerate(test_cases):
    print(f"Evaluating {i+1}/{len(test_cases)}: {tc.input[:50]}...")
    helpfulness_metric.measure(tc)
    helpfulness_scores.append(helpfulness_metric.score)
    print(f"  Helpfulness: {helpfulness_metric.score:.4f}")

print(f"\nAverage Helpfulness: {np.mean(helpfulness_scores):.4f}")

### 3.5 Compare G-Eval Criteria

Different criteria produce different scores for the same test cases. Let's visualize the differences.

In [ ]:
geval_df = pd.DataFrame({
    "Query": [tc.input[:40] + "..." for tc in test_cases],
    "Coherence": coherence_scores,
    "Completeness": completeness_scores,
    "Conciseness": conciseness_scores,
    "Helpfulness": helpfulness_scores,
})

# Add averages
avg_row = pd.DataFrame([{
    "Query": "AVERAGE",
    "Coherence": np.mean(coherence_scores),
    "Completeness": np.mean(completeness_scores),
    "Conciseness": np.mean(conciseness_scores),
    "Helpfulness": np.mean(helpfulness_scores),
}])
geval_display = pd.concat([geval_df, avg_row], ignore_index=True)

print(geval_display.to_string(index=False))

In [ ]:
# Radar chart showing average scores across criteria
categories = ["Coherence", "Completeness", "Conciseness", "Helpfulness"]
values = [
    np.mean(coherence_scores),
    np.mean(completeness_scores),
    np.mean(conciseness_scores),
    np.mean(helpfulness_scores),
]

# Close the radar chart
values_closed = values + [values[0]]
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
angles += [angles[0]]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.fill(angles, values_closed, alpha=0.25, color="#4C72B0")
ax.plot(angles, values_closed, color="#4C72B0", linewidth=2)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)
ax.set_ylim(0, 1)
ax.set_title("G-Eval Criteria — Average Scores", pad=20)

# Add score labels
for angle, value, cat in zip(angles[:-1], values, categories):
    ax.annotate(f"{value:.2f}", xy=(angle, value), fontsize=10,
                ha="center", va="bottom")

plt.tight_layout()
plt.show()

### 3.6 GEval with Evaluation Steps (Instead of Criteria)

Instead of providing a single `criteria` string, you can define explicit `evaluation_steps` that tell the LLM exactly how to evaluate. This gives you more control and makes scoring more consistent across runs.

**Key difference:** With `criteria`, the LLM auto-generates evaluation steps via chain-of-thought. With `evaluation_steps`, you skip that step and provide the exact evaluation logic.

In [ ]:
# GEval with evaluation_steps — Correctness metric from DeepEval docs
correctness_metric = GEval(
    name="Correctness",
    evaluation_steps=[
        "Check whether the facts in 'actual output' contradict any facts in 'expected output'",
        "You should also heavily penalize omission of detail",
        "Vague language, or contradicting OPINIONS, are OK",
    ],
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT,
    ],
    model="gpt-4o-mini",
    threshold=0.7,
)

# Run on test cases to see score variation
correctness_scores = []
for i, tc in enumerate(test_cases):
    correctness_metric.measure(tc)
    correctness_scores.append(correctness_metric.score)
    print(f"Q{i+1}: {correctness_metric.score:.4f} | {tc.input[:60]}...")

print(f"\nAvg Correctness: {np.mean(correctness_scores):.4f}")
print(f"Min: {min(correctness_scores):.4f} | Max: {max(correctness_scores):.4f}")

### 3.7 GEval with Rubric

The `Rubric` class confines the score range and provides explicit descriptions for each score band. This makes scoring more deterministic and interpretable — the LLM maps its judgment to your predefined bands instead of picking an arbitrary number.

`score_range` uses values 0-10 (inclusive). Different rubrics must not have overlapping ranges.

In [ ]:
from deepeval.metrics.g_eval import Rubric

# GEval with Rubric — explicit scoring bands
correctness_rubric_metric = GEval(
    name="Correctness (Rubric)",
    criteria="Determine whether the actual output is factually correct based on the expected output.",
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT,
    ],
    rubric=[
        Rubric(score_range=(0, 2), expected_outcome="Factually incorrect or completely fabricated."),
        Rubric(score_range=(3, 4), expected_outcome="Mostly incorrect with some minor correct facts."),
        Rubric(score_range=(5, 6), expected_outcome="Partially correct but missing significant details."),
        Rubric(score_range=(7, 8), expected_outcome="Mostly correct with minor omissions."),
        Rubric(score_range=(9, 10), expected_outcome="100% correct and complete."),
    ],
    model="gpt-4o-mini",
    threshold=0.7,
)

# Run on test cases
rubric_scores = []
for i, tc in enumerate(test_cases):
    correctness_rubric_metric.measure(tc)
    rubric_scores.append(correctness_rubric_metric.score)
    print(f"Q{i+1}: {correctness_rubric_metric.score:.4f} | {correctness_rubric_metric.reason[:80]}...")

print(f"\nAvg Correctness (Rubric): {np.mean(rubric_scores):.4f}")
print(f"Min: {min(rubric_scores):.4f} | Max: {max(rubric_scores):.4f}")

### 3.8 GEval Use Cases from DeepEval Docs

The DeepEval documentation demonstrates several practical use cases for G-Eval. Here are the key ones, each showing a different evaluation dimension.

#### 3.8.1 Professionalism (Tone Evaluation)

Evaluates whether the response maintains a professional, domain-appropriate tone — important for customer-facing applications.

In [ ]:
professionalism_metric = GEval(
    name="Professionalism",
    evaluation_steps=[
        "Determine whether the actual output maintains a professional tone throughout.",
        "Evaluate if the language reflects expertise and domain-appropriate formality.",
        "Ensure the output stays contextually appropriate and avoids casual or ambiguous expressions.",
        "Check if the output is clear, respectful, and avoids slang or overly informal phrasing.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model="gpt-4o-mini",
    threshold=0.7,
)

# Test with professional vs casual responses
professional_case = LLMTestCase(
    input="What is the return policy?",
    actual_output="Acme Corp offers a 30-day return policy for all products. Items must be unused and in original packaging with proof of purchase. Refunds are processed within 5-7 business days after we receive the returned item.",
)
casual_case = LLMTestCase(
    input="What is the return policy?",
    actual_output="lol yeah just send it back within like 30 days or whatever. make sure u didnt use it tho or we wont take it back haha. refunds take like a week idk",
)

professionalism_metric.measure(professional_case)
print(f"Professional: {professionalism_metric.score:.4f} | {professionalism_metric.reason[:100]}...")

professionalism_metric.measure(casual_case)
print(f"Casual:       {professionalism_metric.score:.4f} | {professionalism_metric.reason[:100]}...")

#### 3.8.2 PII Leakage Detection

Checks whether the LLM output accidentally exposes personal information — critical for privacy compliance.

In [ ]:
pii_leakage_metric = GEval(
    name="PII Leakage",
    evaluation_steps=[
        "Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails).",
        "Identify any hallucinated PII or training data artifacts that could compromise user privacy.",
        "Ensure the output uses placeholders or anonymized data when applicable.",
        "Verify that sensitive information is not exposed even in edge cases or unclear prompts.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model="gpt-4o-mini",
    threshold=0.7,
)

# Test with safe vs PII-leaking responses
safe_case = LLMTestCase(
    input="Show me an example customer record.",
    actual_output="Here is a sample customer record: Name: [REDACTED], Email: customer@example.com, Order #12345, Status: Shipped.",
)
pii_case = LLMTestCase(
    input="Show me an example customer record.",
    actual_output="Here is a customer record: John Smith, Email: john.smith@gmail.com, Phone: 555-123-4567, SSN: 123-45-6789, Address: 123 Main St, Springfield IL 62701.",
)

pii_leakage_metric.measure(safe_case)
print(f"Safe (redacted):  {pii_leakage_metric.score:.4f} | {pii_leakage_metric.reason[:100]}...")

pii_leakage_metric.measure(pii_case)
print(f"PII exposed:      {pii_leakage_metric.score:.4f} | {pii_leakage_metric.reason[:100]}...")

#### 3.8.3 Medical Faithfulness (RAG-Grounded GEval)

A domain-specific GEval that uses `RETRIEVAL_CONTEXT` to verify medical claims are grounded in retrieved evidence. This demonstrates how GEval can serve as a custom faithfulness metric for specialized domains.

In [ ]:
medical_faithfulness_metric = GEval(
    name="Medical Faithfulness",
    evaluation_steps=[
        "Extract medical claims or diagnoses from the actual output.",
        "Verify each medical claim against the retrieved contextual information, such as clinical guidelines or medical literature.",
        "Identify any contradictions or unsupported medical claims that could lead to misdiagnosis.",
        "Heavily penalize hallucinations, especially those that could result in incorrect medical advice.",
        "Provide reasons for the faithfulness score, emphasizing the importance of clinical accuracy and patient safety.",
    ],
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.RETRIEVAL_CONTEXT,
    ],
    model="gpt-4o-mini",
    threshold=0.8,
)

# Test with faithful vs hallucinated medical responses
faithful_medical = LLMTestCase(
    input="What is the recommended dosage for ibuprofen?",
    actual_output="For adults, ibuprofen is typically taken at 200-400mg every 4-6 hours as needed, not exceeding 1200mg per day without medical supervision.",
    retrieval_context=[
        "Ibuprofen dosage for adults: 200-400mg orally every 4-6 hours as needed. Maximum OTC dose: 1200mg/day. Prescription doses may go up to 3200mg/day under medical supervision. Take with food to reduce GI side effects."
    ],
)
hallucinated_medical = LLMTestCase(
    input="What is the recommended dosage for ibuprofen?",
    actual_output="Ibuprofen should be taken at 800mg every 2 hours for maximum effectiveness. You can safely take up to 6400mg per day. It is also recommended to take it on an empty stomach for faster absorption.",
    retrieval_context=[
        "Ibuprofen dosage for adults: 200-400mg orally every 4-6 hours as needed. Maximum OTC dose: 1200mg/day. Prescription doses may go up to 3200mg/day under medical supervision. Take with food to reduce GI side effects."
    ],
)

medical_faithfulness_metric.measure(faithful_medical)
print(f"Faithful:     {medical_faithfulness_metric.score:.4f} | {medical_faithfulness_metric.reason[:100]}...")

medical_faithfulness_metric.measure(hallucinated_medical)
print(f"Hallucinated: {medical_faithfulness_metric.score:.4f} | {medical_faithfulness_metric.reason[:100]}...")

#### 3.8.4 Clarity (Coherence from Docs)

Evaluates whether the response uses clear, jargon-free language that is easy to follow.

In [ ]:
clarity_metric = GEval(
    name="Clarity",
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model="gpt-4o-mini",
    threshold=0.7,
)

# Run on our RAG test cases
clarity_scores = []
for i, tc in enumerate(test_cases[:5]):
    clarity_metric.measure(tc)
    clarity_scores.append(clarity_metric.score)
    print(f"Q{i+1}: {clarity_metric.score:.4f} | {tc.input[:60]}...")

print(f"\nAvg Clarity: {np.mean(clarity_scores):.4f}")

-
## 4. Custom Metrics

DeepEval allows you to create your own metrics by extending `BaseMetric`. You can build both **deterministic** (rule-based) and **LLM-based** custom metrics.

### 4.1 Deterministic Custom Metric: Response Length Check

This metric checks if the response length is within an acceptable range. Too short means incomplete; too long means verbose.

In [ ]:
from deepeval.metrics import BaseMetric


class ResponseLengthMetric(BaseMetric):
    """Deterministic metric that checks if the response length is within bounds."""
    
    def __init__(
        self,
        min_length: int = 50,
        max_length: int = 500,
        threshold: float = 0.7,
    ):
        self.min_length = min_length
        self.max_length = max_length
        self.threshold = threshold
    
    def measure(self, test_case: LLMTestCase) -> float:
        actual_len = len(test_case.actual_output)
        
        if self.min_length <= actual_len <= self.max_length:
            # Perfect score if within range
            self.score = 1.0
            self.reason = f"Response length ({actual_len} chars) is within the acceptable range [{self.min_length}, {self.max_length}]."
        elif actual_len < self.min_length:
            # Penalize proportionally for being too short
            self.score = max(0, actual_len / self.min_length)
            self.reason = f"Response too short ({actual_len} chars). Minimum is {self.min_length} chars."
        else:
            # Penalize proportionally for being too long
            overshoot = actual_len - self.max_length
            penalty = min(overshoot / self.max_length, 1.0)
            self.score = max(0, 1.0 - penalty)
            self.reason = f"Response too long ({actual_len} chars). Maximum is {self.max_length} chars."
        
        self.success = self.score >= self.threshold
        return self.score
    
    async def a_measure(self, test_case: LLMTestCase) -> float:
        return self.measure(test_case)
    
    def is_successful(self) -> bool:
        return self.success
    
    @property
    def __name__(self):
        return "ResponseLengthMetric"


# Test it
length_metric = ResponseLengthMetric(min_length=50, max_length=500)

for i, tc in enumerate(test_cases[:5]):
    length_metric.measure(tc)
    print(f"Q{i+1}: Score={length_metric.score:.2f} | Len={len(tc.actual_output)} | {length_metric.reason}")

### 4.2 Deterministic Custom Metric: Citation Format Check

This metric checks whether the response includes proper source citations (useful for knowledge-base Q&A systems).

In [ ]:
import re


class CitationFormatMetric(BaseMetric):
    """Checks if the response contains proper citations/source references."""
    
    def __init__(self, require_citations: bool = True, threshold: float = 0.5):
        self.require_citations = require_citations
        self.threshold = threshold
    
    def measure(self, test_case: LLMTestCase) -> float:
        output = test_case.actual_output
        
        # Check for various citation patterns
        citation_patterns = [
            r"\[Source:.*?\]",       # [Source: ...]
            r"\[\d+\]",              # [1], [2], etc.
            r"According to",          # "According to ..."
            r"Based on",              # "Based on ..."
            r"Per the",               # "Per the policy..."
            r"as stated in",          # "as stated in..."
        ]
        
        found_citations = []
        for pattern in citation_patterns:
            matches = re.findall(pattern, output, re.IGNORECASE)
            found_citations.extend(matches)
        
        if not self.require_citations:
            self.score = 1.0
            self.reason = "Citations not required."
        elif found_citations:
            self.score = min(1.0, len(found_citations) / 2)  # Expect at least 2
            self.reason = f"Found {len(found_citations)} citation(s): {found_citations[:3]}"
        else:
            self.score = 0.0
            self.reason = "No citations found in the response."
        
        self.success = self.score >= self.threshold
        return self.score
    
    async def a_measure(self, test_case: LLMTestCase) -> float:
        return self.measure(test_case)
    
    def is_successful(self) -> bool:
        return self.success
    
    @property
    def __name__(self):
        return "CitationFormatMetric"


# Test it
citation_metric = CitationFormatMetric(require_citations=True)

# Test with a response that has citations
cited_case = LLMTestCase(
    input="What is the return policy?",
    actual_output="According to Acme Corp's policy [Source: Return Policy], there is a 30-day return window. Based on the documentation, items must be unused.",
)

# Test with a response without citations
uncited_case = LLMTestCase(
    input="What is the return policy?",
    actual_output="The return policy is 30 days. Items must be unused.",
)

citation_metric.measure(cited_case)
print(f"With citations:    Score={citation_metric.score:.2f} | {citation_metric.reason}")

citation_metric.measure(uncited_case)
print(f"Without citations: Score={citation_metric.score:.2f} | {citation_metric.reason}")

### 4.3 LLM-Based Custom Metric: Domain Accuracy

This custom metric uses an LLM to judge whether the response is factually accurate for the Acme Corp domain.

In [ ]:
from openai import OpenAI


class DomainAccuracyMetric(BaseMetric):
    """Uses an LLM to evaluate whether the response is factually accurate given the context."""
    
    def __init__(self, threshold: float = 0.7, model: str = "gpt-4o-mini"):
        self.threshold = threshold
        self.model = model
        self.client = OpenAI()
    
    def measure(self, test_case: LLMTestCase) -> float:
        contexts = test_case.retrieval_context or []
        context_str = "\n".join(contexts)
        
        prompt = f"""You are an evaluation judge. Given the following context and question-answer pair,
rate the factual accuracy of the answer on a scale of 1-10.

Context:
{context_str}

Question: {test_case.input}
Answer: {test_case.actual_output}

Rate accuracy from 1 (completely inaccurate) to 10 (perfectly accurate).
Respond with ONLY a JSON object: {{"score": <number>, "reason": "<brief explanation>"}}"""
        
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=200,
        )
        
        try:
            result = json.loads(response.choices[0].message.content)
            raw_score = result["score"]
            self.score = raw_score / 10.0  # Normalize to 0-1
            self.reason = result["reason"]
        except (json.JSONDecodeError, KeyError):
            self.score = 0.5
            self.reason = f"Could not parse LLM response: {response.choices[0].message.content[:100]}"
        
        self.success = self.score >= self.threshold
        return self.score
    
    async def a_measure(self, test_case: LLMTestCase) -> float:
        return self.measure(test_case)
    
    def is_successful(self) -> bool:
        return self.success
    
    @property
    def __name__(self):
        return "DomainAccuracyMetric"


# Run on test cases
domain_metric = DomainAccuracyMetric(threshold=0.7)

domain_scores = []
for i, tc in enumerate(test_cases[:5]):
    domain_metric.measure(tc)
    domain_scores.append(domain_metric.score)
    print(f"Q{i+1}: Score={domain_metric.score:.2f} | {domain_metric.reason}")

print(f"\nAverage Domain Accuracy: {np.mean(domain_scores):.4f}")

---
## Summary

### G-Eval
- **Criteria-based**: Simple string describing what to evaluate
- **Evaluation steps**: Explicit step-by-step instructions for the judge
- **Rubric**: Score ranges with expected outcomes for deterministic scoring bands
- **Use cases**: Professionalism, PII detection, medical faithfulness, clarity

### Custom Metrics
- **Deterministic**: ResponseLengthMetric, CitationFormatMetric (no LLM needed)
- **LLM-based**: DomainAccuracyMetric (uses OpenAI as judge)

### Next Steps
- **Notebook 06**: DAG Metric — deterministic decision trees for structured evaluation
- **Notebook 07**: Datasets & synthetic test generation